# M4-L3 — Coarse-grained Monte Carlo: measuring what L1 assumes

**BruSafe / CAPSID · iGEM 2026 · Module 4 — Assembly**

This notebook is self-contained: the first code cell writes the simulation source to disk, so nothing has to be downloaded. It runs on Colab, in Jupyter, or on a laptop with `numpy`, `scipy` and `matplotlib`.

The theory and the bench protocols (§11) are described in [`M4_theory_compendium.pdf`](../../docs/M4_theory_compendium.pdf) in `capsid/docs/`.

**What this layer does.** The contact ladder $c(j)$ of L1, that is, the number of new bonds a subunit makes when it joins a shell of size $j$, is an *assumption*. L3 measures it. Ninety patchy spheres with five curved patches (the interface geometry of the T=3 dimer) are sampled by NVT Metropolis Monte Carlo, and the bond census of the resulting clusters gives the ladder that L1 should have used.

**This is the slowest layer.** The defaults below are reduced so that it runs on a laptop; the settings of the reference scan are given in the comments.

In [ ]:
# Colab / fresh machine: uncomment
# !pip -q install numpy scipy matplotlib
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

In [ ]:
%%writefile brusafe_cg_assembly.py
"""
BruSafe Module 4 - Coarse-Grained Capsid Assembly (patchy-particle Metropolis MC)
==================================================================================
A spatial, orientation-resolved coarse-grained model of MS2 coat-protein-dimer
self-assembly. Each subunit is one rigid patchy particle (the coat-protein dimer,
the assembly unit established in Module 1). Binding is orientation-dependent
(Kern-Frenkel patches) with a curvature-encoding geometry so that growing sheets
close into a shell. Pure numpy + NVT Metropolis Monte Carlo - no GPU, no MD engine.

This is the SPATIAL companion to the mean-field nucleation-elongation ODEs of
Module 4. It independently reproduces the kinetic trap: assembly QUALITY (fraction
of well-coordinated subunits) is non-monotonic in subunit concentration, peaking
at intermediate packing fraction and collapsing when crowding spawns competing
nuclei that cannot anneal. Cross-scale agreement between two models built on
different assumptions (well-mixed rate equations vs explicit diffusing bodies) is
the validation argument.

Physics / parameter provenance
------------------------------
- Dimer = assembly subunit:            Ni et al. 1995 (10.1016/s0969-2126(01)00156-3)
- Patchy-particle / Kern-Frenkel CG:   Perlmutter & Hagan 2015 (10.1146/annurev-physchem-040214-121637)
- Nucleation-elongation, kinetic trap: Endres & Zlotnick 2002; Hagan 2014
- Bond free energy scale (eps ~ kT):   Ceres & Zlotnick 2002 (dimer-dimer ~ -4 kcal/mol -> ~7 kT at 310 K)

Reduced units: subunit diameter SIGMA = 1 (~5 nm for a dimer); energies in kT.
"""
import numpy as np
from collections import Counter

# ---- geometry constants -----------------------------------------------------
SIGMA   = 1.0                       # subunit diameter (reduced units)
R_WELL  = 1.20 * SIGMA              # attractive-well outer edge
PATCH_HALFWIDTH_DEG = 28.0          # Kern-Frenkel angular tolerance of an interface
COS_TOL = np.cos(np.radians(PATCH_HALFWIDTH_DEG))

# ---- subunit patch geometry -------------------------------------------------
def make_patches(kind="curved5"):
    """Body-frame unit vectors for the binding patches, tilted onto a cone of
    half-angle (90 - wedge) so each subunit prefers an out-of-plane bond angle;
    that built-in curvature makes flat sheets close into a finite shell."""
    if kind == "curved5":   n, wedge = 5, 20.0
    elif kind == "curved6": n, wedge = 6, 12.0
    else:                   n, wedge = int(kind[-1]), 18.0
    tilt = np.radians(90.0 - wedge)
    az = np.linspace(0, 2*np.pi, n, endpoint=False)
    P = np.stack([np.sin(tilt)*np.cos(az), np.sin(tilt)*np.sin(az),
                  np.cos(tilt)*np.ones(n)], axis=1)
    return P / np.linalg.norm(P, axis=1, keepdims=True)

# ---- rotation helpers -------------------------------------------------------
def rand_rot(rng):
    u1,u2,u3 = rng.random(3)
    q = np.array([np.sqrt(1-u1)*np.sin(2*np.pi*u2), np.sqrt(1-u1)*np.cos(2*np.pi*u2),
                  np.sqrt(u1)*np.sin(2*np.pi*u3),  np.sqrt(u1)*np.cos(2*np.pi*u3)])
    w,x,y,z = q
    return np.array([[1-2*(y*y+z*z),2*(x*y-w*z),2*(x*z+w*y)],
                     [2*(x*y+w*z),1-2*(x*x+z*z),2*(y*z-w*x)],
                     [2*(x*z-w*y),2*(y*z+w*x),1-2*(x*x+y*y)]])

def small_rot(rng, sigma_ang):
    ax = rng.normal(size=3); ax/=np.linalg.norm(ax)
    th = rng.normal()*sigma_ang
    c,s = np.cos(th),np.sin(th); x,y,z = ax; C=1-c
    return np.array([[c+x*x*C,x*y*C-z*s,x*z*C+y*s],
                     [y*x*C+z*s,c+y*y*C,y*z*C-x*s],
                     [z*x*C-y*s,z*y*C+x*s,c+z*z*C]])

# ---- pair energy (hard core + Kern-Frenkel patch + curvature reward) --------
def pair_energy(dr, r, Ri, Rj, P, eps):
    if r < SIGMA:   return np.inf          # excluded volume
    if r > R_WELL:  return 0.0
    dhat = dr / r
    pi = Ri @ P.T; pj = Rj @ P.T
    ci = pi.T @ dhat; cj = pj.T @ (-dhat)
    if ci.max() > COS_TOL and cj.max() > COS_TOL:
        ang  = pi[:, ci.argmax()] @ (-pj[:, cj.argmax()])   # correct dihedral -> anti-aligned
        curv = 0.5*(1+ang)
        return -eps*(0.5 + 0.5*curv)
    return 0.0

# ---- cell list --------------------------------------------------------------
def build_cells(pos, L, rc):
    n = max(1, int(L/rc)); cs = L/n
    idx = np.floor(pos/cs).astype(int) % n
    cells = {}
    for i,(a,b,c) in enumerate(idx):
        cells.setdefault((a,b,c),[]).append(i)
    return cells, n, cs

def neighbors(i, pos, cells, n, cs, L):
    a,b,c = (pos[i]//cs).astype(int) % n
    out=[]
    for da in (-1,0,1):
        for db in (-1,0,1):
            for dc in (-1,0,1):
                out += cells.get(((a+da)%n,(b+db)%n,(c+dc)%n),[])
    return [j for j in out if j!=i]

def _pe(i,pos,R,P,eps,nbrs,L):
    e=0.0
    for j in nbrs:
        dr=pos[j]-pos[i]; dr-=L*np.round(dr/L); r=np.sqrt(dr@dr)
        if r<R_WELL: e+=pair_energy(dr,r,R[i],R[j],P,eps)
    return e

# ---- order parameters -------------------------------------------------------
def quality(pos,R,P,eps,L):
    """(largest_cluster, well_ordered_fraction, mean_coordination).
    A bond requires energy < -0.4*eps (a geometrically correct interface, not a
    grazing contact). well_ordered_fraction = fraction with coordination >= 3,
    i.e. subunits locked into a shell fragment rather than a dangling edge."""
    N=len(pos); cells,ncell,cs=build_cells(pos,L,R_WELL)
    coord=np.zeros(N,int); parent=list(range(N))
    def find(x):
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for i in range(N):
        for j in neighbors(i,pos,cells,ncell,cs,L):
            if j<=i: continue
            dr=pos[j]-pos[i]; dr-=L*np.round(dr/L); r=np.sqrt(dr@dr)
            if r<R_WELL and pair_energy(dr,r,R[i],R[j],P,eps)<-0.4*eps:
                coord[i]+=1; coord[j]+=1
                a,b=find(i),find(j)
                if a!=b: parent[a]=b
    sizes=Counter(find(i) for i in range(N))
    return max(sizes.values()), float((coord>=3).mean()), float(coord.mean())

# ---- MC driver --------------------------------------------------------------
def run_mc(N, phi, eps, patches="curved5", sweeps=1200, seed=0,
           trans_step=0.35, d_rot=0.45, record_every=100):
    """NVT Metropolis MC of N patchy subunits at packing fraction phi and bond
    energy eps (units of kT). Returns (trajectory, pos, R, L) where trajectory
    rows are (sweep, largest_cluster, well_ordered_fraction, mean_coordination)."""
    rng = np.random.default_rng(seed)
    P = make_patches(patches)
    L = (N*(np.pi/6)*SIGMA**3 / phi)**(1/3)
    pos = rng.random((N,3))*L
    R = np.stack([rand_rot(rng) for _ in range(N)])
    traj=[]
    for sw in range(sweeps):
        cells,ncell,cs = build_cells(pos,L,R_WELL)
        for i in rng.permutation(N):
            nbrs = neighbors(i,pos,cells,ncell,cs,L)
            e_old=_pe(i,pos,R,P,eps,nbrs,L)
            old_p=pos[i].copy(); old_R=R[i].copy()
            if rng.random()<0.5:
                pos[i]=(pos[i]+(rng.random(3)-0.5)*trans_step)%L
            else:
                R[i]=small_rot(rng,d_rot)@R[i]
            e_new=_pe(i,pos,R,P,eps,nbrs,L)
            de=e_new-e_old
            if np.isnan(de): accept=(e_new!=np.inf)          # overlap->overlap
            else:            accept=(de<=0) or (rng.random()<np.exp(-min(de,700)))
            if not accept: pos[i]=old_p; R[i]=old_R
        if sw%record_every==0 or sw==sweeps-1:
            traj.append((sw,)+quality(pos,R,P,eps,L))
    return np.array(traj), pos, R, L

# ---- demonstration: the kinetic trap (concentration scan) -------------------
if __name__ == "__main__":
    print("Concentration scan (Mode A prediction, no wet-lab data):")
    print(" phi   well-ordered   mean-coord   largest")
    for phi in [0.03,0.06,0.10,0.15,0.22,0.30]:
        q=[run_mc(N=90,phi=phi,eps=11.0,sweeps=1200,seed=s,record_every=1200)[0][-1]
           for s in range(3)]
        q=np.array(q)
        print(f" {phi:.2f}   {q[:,2].mean():.3f}         {q[:,3].mean():.2f}"
              f"         {q[:,1].mean():.0f}")
    print("\nPrediction: quality is non-monotonic in concentration - the kinetic")
    print("trap, reproduced independently from the Module 4 mean-field ODEs.")


## 1. One trajectory

`run_mc` returns a trajectory whose rows are `(sweep, largest_cluster, well_ordered_fraction, mean_coordination)`. A subunit counts as well-ordered when it has ≥3 satisfied contacts.

In [ ]:
import brusafe_cg_assembly as L3, time

t0 = time.time()
traj, pos, R, L = L3.run_mc(N=90, phi=0.15, eps=11.0, sweeps=400, seed=0, record_every=25)
traj = np.array(traj)
print(f'{len(traj)} recorded points in {time.time()-t0:.0f} s')
print(f'final: largest cluster {traj[-1,1]:.0f}, well-ordered {traj[-1,2]:.2f}, '
      f'mean coordination {traj[-1,3]:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
for ax, k, lab, col in zip(axes, [1,2,3],
        ['largest cluster (subunits)','well-ordered fraction','mean coordination'],
        ['#0b6e5a','#8a4f7d','#c8863c']):
    ax.plot(traj[:,0], traj[:,k], lw=1.8, color=col)
    ax.set_xlabel('MC sweep'); ax.set_ylabel(lab)
fig.suptitle(r'CG assembly trajectory, $\phi$=0.15, $\varepsilon$=11 kT', y=1.02)
fig.tight_layout(); fig.savefig('M4_L3_trajectory.png', dpi=200, bbox_inches='tight')

## 2. Volume-fraction scan

The reference scan (described in the theory compendium) used `sweeps=1200`, 3 seeds and six φ values, and took roughly 3½ minutes of wall time. The cell below runs a reduced version by default; the reference settings can be restored by uncommenting the line indicated.

The quantity of interest is the well-ordered fraction, which is itself peaked in φ. This is the coarse-grained counterpart of the concentration optimum of L1, reached by a completely different route.

In [ ]:
PHIS   = [0.03, 0.06, 0.10, 0.15, 0.22, 0.30]
SWEEPS, NSEED = 300, 2          # reduced for a laptop
# SWEEPS, NSEED = 1200, 3       # reference settings (~3.5 min)

print(' phi   well-ordered   mean-coord   largest')
scan = []
for phi in PHIS:
    q = np.array([L3.run_mc(N=90, phi=phi, eps=11.0, sweeps=SWEEPS, seed=s,
                            record_every=SWEEPS)[0][-1] for s in range(NSEED)])
    scan.append((phi, q[:,2].mean(), q[:,3].mean(), q[:,1].mean()))
    print(f' {phi:.2f}      {q[:,2].mean():.3f}        {q[:,3].mean():.2f}      {q[:,1].mean():.1f}')
scan = np.array(scan)

> **Convergence warning.** The defaults above are reduced so that the notebook finishes on a laptop. At `SWEEPS = 200–400` the scan does *not* reproduce the non-monotonic optimum of the reference scan: it rises monotonically, because the crowded boxes have not yet had time to become trapped. The kinetic trap is a late-time phenomenon, so `SWEEPS = 1200` and `NSEED = 3` (~200 s) should be set before any number from this cell is quoted.

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.9))
ax.plot(scan[:,0], scan[:,1], 'o-', lw=2, color='#0b6e5a', label='well-ordered fraction')
ax2 = ax.twinx()
ax2.plot(scan[:,0], scan[:,3], 's--', lw=1.6, color='#c8863c', label='largest cluster')
ax.set_xlabel(r'volume fraction $\phi$'); ax.set_ylabel('well-ordered fraction')
ax2.set_ylabel('largest cluster (subunits)')
ax.set_title('CG optimum in volume fraction')
h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, frameon=False, fontsize=9, loc='upper left')
fig.tight_layout(); fig.savefig('M4_L3_phi_scan.png', dpi=200)
pk = scan[int(np.argmax(scan[:,1])), 0]
print(f'peak well-ordered fraction at phi = {pk:.2f}')
print()
print('REFERENCE SCAN (SWEEPS=1200, NSEED=3, ~200 s): peak at phi = 0.10-0.15,')
print('well-ordered fraction rising to 0.39 and falling back to 0.21 by phi = 0.30.')
if pk >= 0.22:
    print()
    print('>> This run peaked at high phi, which indicates under-convergence and not a result:')
    print('   at low sweep counts the dense boxes have simply had more collisions,')
    print('   and the crowded systems have not yet had time to fail to anneal.')
    print('   The curve should not be quoted; SWEEPS should be raised to 1200 and NSEED to 3.')

---
**Outputs:** `M4_L3_trajectory.png`, `M4_L3_phi_scan.png`.

`M4_bridge_CG_to_ODE_to_bench.ipynb` turns the bond census of these clusters into a contact ladder.